# DHL Pickup Optimization: Customer Segmentation using RFM + K-Means Clustering
**Topic: Unsupervised Learning – Clustering (K-Means)**

In [ ]:
# --- Import Libraries ---
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
# --- Load the Dataset ---
# This dataset simulates DHL customer pickup transactions
df = pd.read_excel("Online Retail.xlsx")  # Replace with your dataset path if needed

In [ ]:
# --- Preprocessing ---
df = df.dropna(subset=['CustomerID'])
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

In [ ]:
# --- RFM Feature Engineering ---
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

In [ ]:
# --- Handle Outliers with Log Transform ---
rfm['Monetary_log'] = np.log1p(rfm['Monetary'])
rfm['Frequency_log'] = np.log1p(rfm['Frequency'])

In [ ]:
# --- Standardize Features ---
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency_log', 'Monetary_log']])

In [ ]:
# --- Apply K-Means Clustering ---
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
# --- Visualize the Clusters ---
plt.figure(figsize=(8, 6))
plt.scatter(rfm['Recency'], rfm['Monetary_log'], c=rfm['Cluster'], cmap='viridis')
plt.title("DHL Customer Segments based on RFM Clustering")
plt.xlabel("Recency (days since last pickup)")
plt.ylabel("log(Monetary Value)")
plt.grid(True)
plt.colorbar(label='Cluster ID')
plt.show()

### 🔍 Elbow Method for Choosing Optimal Number of Clusters
The Elbow Method helps us find the optimal number of clusters (k) for K-Means.
It does this by plotting the **Within-Cluster Sum of Squares (WCSS)** for different values of k.
The 'elbow point' on the curve (where the rate of decrease sharply slows) suggests the best k.

In [ ]:
# --- Elbow Method to Determine Optimal k ---
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.grid(True)
plt.show()